# vgscp — full REAL multi-seed run (E1–E4) on Colab GPU

Runs the four experiments end-to-end on **real** Waterbirds + CUB-200 data with frozen CLIP ViT-B/32 features (encoded once, cached) and logistic heads — **no large-model training**. Run top-to-bottom on a **GPU** runtime (`Runtime → Change runtime type → GPU`).

Honesty: the pre-committed verdicts (`eval/e1_verdict.py`, `eval/scacp_gate.py`) are LOCKED. Whatever the real multi-seed numbers are — including FALLBACK / ties / softenings — is what gets reported.

## 0. Parameters — **EDIT THESE**

In [ ]:
# ===================== EDIT THESE =====================
REPO_SOURCE    = "git"          # "git" or "drive"
REPO_URL       = "https://github.com/<YOUR_USER>/vgscp.git"   # EDIT (private: https://<TOKEN>@github.com/<user>/vgscp.git)
REPO_BRANCH    = "main"
REPO_DRIVE_ZIP = "/content/drive/MyDrive/vgscp.zip"           # used only if REPO_SOURCE=="drive"

DRIVE_CACHE    = "/content/drive/MyDrive/vgscp_cache"  # datasets + CLIP feature cache persisted here
SEEDS          = 10                                    # >=10 random cal/test splits per spec

# Dataset URLs — EDIT IF URL CHANGES
WATERBIRDS_URL = "https://nlp.stanford.edu/data/dro/waterbird_complete95_forest2water2.tar.gz"
CUB_URL        = "https://data.caltech.edu/records/65de6-vp158/files/CUB_200_2011.tgz"
# ======================================================
import os, time, subprocess, sys
def sh(cmd, **kw):
    print("$", cmd); return subprocess.run(cmd, shell=True, **kw)
def run_module(mod_args, label):
    t = time.time(); print(f"\n===== {label} =====")
    p = subprocess.run([sys.executable, "-m", *mod_args])
    dt = time.time() - t; print(f"[{label}] exit={p.returncode}  wall={dt/60:.1f} min")
    if dt > 4.5 * 3600: print(f"[WARN] {label} approaching the 5h cap")
    return p.returncode

## 1. GPU check + install

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("[WARN] No GPU — CLIP encode will be slow. Set Runtime->GPU.")
import subprocess
subprocess.run("pip -q install open_clip_torch ftfy regex tqdm pyyaml scikit-learn pandas matplotlib", shell=True)

## 2. Mount Drive (persist datasets + CLIP feature cache across restarts)

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_CACHE, exist_ok=True)
print("cache dir:", DRIVE_CACHE)

## 3. Get the repo

In [ ]:
REPO_DIR = "/content/vgscp"
if REPO_SOURCE == "git":
    sh(f"rm -rf {REPO_DIR} && git clone --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}")
else:
    sh(f"rm -rf {REPO_DIR} && mkdir -p {REPO_DIR} && unzip -q {REPO_DRIVE_ZIP} -d {REPO_DIR}")
    subs = [d for d in os.listdir(REPO_DIR) if os.path.isdir(f"{REPO_DIR}/{d}")]
    if len(subs) == 1 and not os.path.exists(f"{REPO_DIR}/scripts"):
        inner = f"{REPO_DIR}/{subs[0]}"; sh(f"shopt -s dotglob && mv {inner}/* {REPO_DIR}/ && rmdir {inner}")
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)
print("repo:", os.getcwd())

## 4. Datasets — download (cached to Drive) + extract, set env vars

In [ ]:
def fetch(url, drive_name, extract_to):
    os.makedirs(extract_to, exist_ok=True)
    tarball = os.path.join(DRIVE_CACHE, drive_name)
    if not os.path.exists(tarball):
        sh(f"wget -q -O '{tarball}' '{url}'")
    else:
        print("cached tarball:", tarball)
    sh(f"tar -xzf '{tarball}' -C '{extract_to}'")
    return extract_to

fetch(WATERBIRDS_URL, "waterbirds.tar.gz", "/content/data/waterbirds")
fetch(CUB_URL, "CUB_200_2011.tgz", "/content/data/cub")
os.environ["WATERBIRDS_ROOT"] = "/content/data/waterbirds"
os.environ["CUB_ROOT"] = "/content/data/cub"
# point the repo CLIP feature cache at Drive so re-runs skip re-encoding
sh("rm -rf results/cache_clip"); os.makedirs("results", exist_ok=True)
os.makedirs(f"{DRIVE_CACHE}/clip", exist_ok=True)
sh(f"ln -s {DRIVE_CACHE}/clip results/cache_clip")
print("WATERBIRDS_ROOT=", os.environ["WATERBIRDS_ROOT"]); print("CUB_ROOT=", os.environ["CUB_ROOT"])

## 5. Encode + cache CLIP features ONCE (later cells reuse the cache)

In [ ]:
t = time.time()
from config_util import load_config
from experiments.real_data import load_real_bundle
cfg = load_config("configs/cub200_frontier.yaml")
bundle = load_real_bundle(cfg, seed=0)
print("feature shapes:", {k: v.shape for k, v in bundle.features.items()})
print("species present:", bundle.info["n_species_present"], "| CUB join:", bundle.info["cub_join"]["coverage"])
print(f"[encode+cache] wall={(time.time()-t)/60:.1f} min (cached to Drive)")

## 6. E1 — CUB-200 multiclass frontier

In [ ]:
run_module(["scripts.run_cub200_frontier", "--config", "configs/cub200_frontier.yaml", "--seeds", str(SEEDS)], "E1")
import pandas as pd, json
from IPython.display import Image, display
print(json.load(open("results/e1/e1_results.json"))["verdicts"]["APS"]["rationale"])
df = pd.read_csv("results/e1/e1_frontier.csv")
g = (df[df.score == "APS"].groupby(["scheme", "test_corr"])
     .agg(worst_cov=("worst_cov", "mean"), set_size=("mean_set_size", "mean"), gap=("cov_gap", "mean")).round(3))
display(g)
for s in ("APS", "RAPS", "THR"):
    p = f"results/figures/e1_frontier_{s}.png"
    if os.path.exists(p): display(Image(p))

## 7. E3 — correlation-strength shift (worst-group coverage + gap)

In [ ]:
run_module(["scripts.run_e3_shift_multiseed", "--config", "configs/shiftcp_derisk.yaml", "--seeds", str(SEEDS)], "E3")
import pandas as pd
d = pd.read_csv("results/e3/e3_shift_metrics.csv")
display(d.groupby(["score", "scheme", "rho_test"])
        .agg(worst_group_cov=("worst_group_cov", "mean"), cov_gap=("cov_gap", "mean")).round(3))

## 8. E2 — verifiability collapse (clean vs mixed; minority + contamination AUROC)

In [ ]:
run_module(["scripts.run_e2_verifiability_multiseed", "--config", "configs/premise2_waterbirds.yaml", "--seeds", str(SEEDS)], "E2")
import pandas as pd
d = pd.read_csv("results/e2/e2_verifiability_metrics.csv")
display(d.groupby(["space", "signal"])
        .agg(min_auroc=("minority_auroc", "mean"), contam=("contamination_auroc", "mean")).round(3))

## 9. E4 — scacp 312-attribute locked-gate scan

In [ ]:
run_module(["scripts.run_e4_scacp_gate", "--real", "--config", "configs/cub200_frontier.yaml"], "E4")
import json, pandas as pd
r = json.load(open("results/e4/e4_results.json"))
print(f"GATE PASSES: {r['n_pass']}/{r['n_attributes']}  median diff-noise AUROC={r['median_diff_auroc']:.3f}")
print(f"per-criterion: diff>=0.70:{r['pass_diff']}  gap>=0.03:{r['pass_gap']}  support>=100:{r['pass_support']}")
d = pd.read_csv("results/e4/e4_scacp_gate_scan.csv")
display(d["diff_noise_auroc"].describe().round(3))

## 10. Consolidate + copy to Drive + zip for download

In [ ]:
import json, datetime, platform
summary = {"date": str(datetime.date.today()), "seeds": SEEDS, "platform": platform.platform()}
for e in ("e1", "e2", "e3", "e4"):
    p = f"results/{e}/{e}_results.json"
    if os.path.exists(p): summary[e] = json.load(open(p))
open("results/REAL_RUN_SUMMARY.json", "w").write(json.dumps(summary, indent=2, default=str))
out = f"{DRIVE_CACHE}/results_real"
sh(f"rm -rf {out} && cp -r results {out}")
for f in ("RESULTS.md", "BLOCKERS.md", "E1_REPORT.md", "E2_REPORT.md", "E3_REPORT.md", "E4_REPORT.md"):
    if os.path.exists(f): sh(f"cp {f} {out}/ 2>/dev/null || true")
sh(f"cd {DRIVE_CACHE} && zip -qr results_real.zip results_real")
print("Saved to Drive:", out, "and", f"{DRIVE_CACHE}/results_real.zip")
try:
    from google.colab import files; files.download(f"{DRIVE_CACHE}/results_real.zip")
except Exception as e: print("download skipped:", e)
print("\nNOTE: regenerate RESULTS.md from the real E*_REPORT.md numbers per the Part-C honesty constraints.")